<a href="https://colab.research.google.com/github/babupallam/fiftyone-visual-agents-lab/blob/feature%2Fvisualsimilarysearch/notebooks/Visual_Similarity_Search_Demo_with_FiftyOne_%2B_CLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Project Title:
# Lightweight Visual Similarity Search Demo with FiftyOne + CLIP
# ============================================================
#
# What this project demonstrates:
# 1) Load a small public image dataset
# 2) Build CLIP embeddings safely in Google Colab
# 3) Visualize similar images
# 4) Create a 2D embedding view
# 5) Query by example using one image
#
# Why this is better for Colab:
# - uses a much smaller model than the one that crashed
# - avoids loading very large checkpoint shards
# - avoids heavy patch-level pipelines
# - still gives a strong workshop/demo-style result
#


# STEP 1) Install required packages

- Installs the libraries needed for this demo.
  - `fiftyone` is used to manage, inspect, and visualize the dataset.
  - `umap-learn` supports dimensionality reduction for embedding visualization.
  - `ftfy` is a text-fixing dependency often required by CLIP-related tooling.
- We do this first so the notebook has all required dependencies before any imports or processing.



In [2]:
# ------------------------------------------------------------
# STEP 1) Install required packages
# ------------------------------------------------------------

!pip -q install fiftyone umap-learn ftfy



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 52.2 MB/s eta 0:00:00

# STEP 2) Import libraries

- Imports all Python packages that will be used throughout the project.
- Keeps the notebook organized by collecting dependencies in one place.
- We do this early so all later steps can use these modules directly.



In [3]:
# ------------------------------------------------------------
# STEP 2) Import libraries
# ------------------------------------------------------------

import os
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.brain as fob
import numpy as np



/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


# STEP 3) Optional cleanup

- Removes any old dataset with the same name from previous notebook runs.
- Prevents conflicts, duplicated datasets, or stale results from earlier executions.
- We do this to make the notebook repeatable and easier to rerun safely.



In [4]:
# ------------------------------------------------------------
# STEP 3) Optional cleanup
# ------------------------------------------------------------
#
# This helps avoid conflicts if the notebook is re-run multiple times.

dataset_name = "clip_similarity_demo"

if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)



# STEP 4) Load a lightweight public dataset

- Loads a small public dataset that is suitable for demonstration in Colab.
- Uses a limited number of samples to keep memory usage low and runtime stable.
- We do this because a lightweight dataset is enough to demonstrate similarity search without crashing the session.



In [5]:
# ------------------------------------------------------------
# STEP 4) Load a lightweight public dataset
# ------------------------------------------------------------
#
# CIFAR-10 is small and safe for Colab.
# We only take a subset to keep the notebook responsive.

dataset = foz.load_zoo_dataset(
    "cifar10",
    split="test",
    max_samples=500,
    dataset_name=dataset_name,
    shuffle=True,
)

print(dataset)
print("Number of samples:", len(dataset))



INFO:fiftyone.zoo.datasets:Downloading split 'test' to '/root/fiftyone/cifar10/test'
100%|██████████| 170M/170M [00:13<00:00, 12.7MB/s]


 100% |█████████████| 10000/10000 [4.8s elapsed, 0s remaining, 1.9K samples/s]      


INFO:eta.core.utils: 100% |█████████████| 10000/10000 [4.8s elapsed, 0s remaining, 1.9K samples/s]      


Dataset info written to '/root/fiftyone/cifar10/info.json'


INFO:fiftyone.zoo.datasets:Dataset info written to '/root/fiftyone/cifar10/info.json'


...

 100% |█████████████████| 500/500 [339.6ms elapsed, 0s remaining, 1.5K samples/s]     


INFO:eta.core.utils: 100% |█████████████████| 500/500 [339.6ms elapsed, 0s remaining, 1.5K samples/s]     


Dataset 'clip_similarity_demo' created


INFO:fiftyone.zoo.datasets:Dataset 'clip_similarity_demo' created


Name:        clip_similarity_demo
Media type:  image
Num samples: 500
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
Number of samples: 500


# STEP 5) Launch FiftyOne App

- Opens the interactive FiftyOne interface for exploring the dataset visually.
- Lets you inspect images, labels, views, and later similarity results.
- We do this so the project can be demonstrated in an interactive and user-friendly way.



In [6]:
# ------------------------------------------------------------
# STEP 5) Launch FiftyOne App
# ------------------------------------------------------------
#
# This opens the interactive dataset viewer.
# In Colab, this should be much safer than the earlier project.

session = fo.launch_app(dataset, auto=False)


Could not connect session, trying again in 10 seconds

Session launched. Run `session.show()` to open the App in a cell output.


INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.



Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.14.1

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



INFO:fiftyone.core.session.session:
Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.14.1

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



# STEP 6) Load a small CLIP model from FiftyOne Zoo

- Loads a lightweight CLIP model for generating image embeddings.
- CLIP converts images into vector representations that capture semantic visual meaning.
- We do this because embeddings are the foundation of similarity search.



In [7]:
# ------------------------------------------------------------
# STEP 6) Load a small CLIP model from FiftyOne Zoo
# ------------------------------------------------------------
#
# This is much lighter than very large embedding models.
# It is a better choice for demonstration in Colab.

model = foz.load_zoo_model("clip-vit-base32-torch")



INFO:fiftyone.core.models:Downloading model from 'https://openaipublic.azureedge.net/clip/models/40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af/ViT-B-32.pt'...


 100% |██████|    2.6Gb/2.6Gb [8.8s elapsed, 0s remaining, 380.5Mb/s]       


INFO:eta.core.utils: 100% |██████|    2.6Gb/2.6Gb [8.8s elapsed, 0s remaining, 380.5Mb/s]       


INFO:fiftyone.utils.clip.zoo:Downloading CLIP tokenizer...


 100% |█████|   10.4Mb/10.4Mb [10.6ms elapsed, 0s remaining, 976.0Mb/s]      


INFO:eta.core.utils: 100% |█████|   10.4Mb/10.4Mb [10.6ms elapsed, 0s remaining, 976.0Mb/s]      


# STEP 7) Compute image embeddings

- Generates embedding vectors for every image in the dataset.
- Stores the vectors in a dataset field for later reuse.
- We do this so each image can be compared numerically based on visual content.


In [8]:
# ------------------------------------------------------------
# STEP 7) Compute image embeddings
# ------------------------------------------------------------
#
# These embeddings represent the visual content of each image.
# We store them in the field "clip_embedding".

dataset.compute_embeddings(
    model,
    embeddings_field="clip_embedding",
)

print("Embeddings computed successfully")



 100% |█████████████████| 500/500 [2.3m elapsed, 0s remaining, 3.0 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 500/500 [2.3m elapsed, 0s remaining, 3.0 samples/s]      


Embeddings computed successfully



# STEP 8) Build similarity index

- Creates a searchable index from the stored embeddings.
- Makes nearest-neighbor search fast and efficient.
- We do this so similar images can be retrieved quickly during the demo.



In [9]:
# ------------------------------------------------------------
# STEP 8) Build similarity index
# ------------------------------------------------------------
#
# This allows us to find visually similar images efficiently.

fob.compute_similarity(
    dataset,
    embeddings="clip_embedding",
    brain_key="clip_similarity",
)

print("Similarity index created successfully")



Similarity index created successfully


# STEP 9) Build 2D visualization

- Reduces high-dimensional embeddings into a 2D representation.
- Makes it easier to visually inspect clusters and relationships between images.
- We do this to provide an intuitive visual understanding of the embedding space.



In [10]:
# ------------------------------------------------------------
# STEP 9) Build 2D visualization
# ------------------------------------------------------------
#
# This reduces the embedding vectors into 2D for exploration.

fob.compute_visualization(
    dataset,
    embeddings="clip_embedding",
    method="umap",
    brain_key="clip_umap",
)

print("2D visualization created successfully")



Generating visualization...


INFO:fiftyone.brain.visualization:Generating visualization...


UMAP( verbose=True)
Fri Apr 10 00:53:16 2026 Construct fuzzy simplicial set
Fri Apr 10 00:53:16 2026 Finding Nearest Neighbors
Fri Apr 10 00:53:22 2026 Finished Nearest Neighbor Search
Fri Apr 10 00:53:28 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Fri Apr 10 00:53:32 2026 Finished embedding
2D visualization created successfully


# STEP 10) Pick one query image

- Selects one sample image to act as the query example.
- Gives a clear starting point for similarity retrieval.
- We do this so the next steps can demonstrate query-by-example search.



In [11]:
# ------------------------------------------------------------
# STEP 10) Pick one query image
# ------------------------------------------------------------
#
# We select the first sample as the query example.

query_sample = dataset.first()
print("Query sample ID:", query_sample.id)
print("Query label:", query_sample.ground_truth.label)



Query sample ID: 69d849327cb6983a1f2fc489
Query label: ship


# STEP 11) Retrieve nearest neighbors

- Finds the most visually similar images to the selected query image.
- Uses the similarity index built from CLIP embeddings.
- We do this to demonstrate the core functionality of image similarity search.



In [12]:
# ------------------------------------------------------------
# STEP 11) Retrieve nearest neighbors
# ------------------------------------------------------------
#
# This finds visually similar images using the similarity index.

similarity_index = dataset.load_brain_results("clip_similarity")

neighbors = similarity_index.sort_by_similarity(
    query_sample.id,
    k=25,
    dist_field="distance",
)

print("Nearest neighbors view created")



Nearest neighbors view created


# STEP 12) Send neighbors to the App for inspection

- Updates the FiftyOne App view to show only the retrieved similar images.
- Makes the retrieval result easy to inspect visually.
- We do this so the demo audience can clearly see the search output.



In [13]:
# ------------------------------------------------------------
# STEP 12) Send neighbors to the App for inspection
# ------------------------------------------------------------
#
# This makes the visual result easy to demonstrate live.

session.view = neighbors



# STEP 13) Show query details in notebook

- Prints basic information about the chosen query image.
- Helps connect the notebook output with what is shown in the visual app.
- We do this to make the query easier to identify and explain.



In [14]:
# ------------------------------------------------------------
# STEP 13) Show query details in notebook
# ------------------------------------------------------------

print("Query image filepath:", query_sample.filepath)
print("Query image class:", query_sample.ground_truth.label)



Query image filepath: /root/fiftyone/cifar10/test/data/003764.jpg
Query image class: ship


# STEP 14) Create a filtered view for one class

- Creates a subset of the dataset containing only one selected class.
- Demonstrates how filtering and views work in FiftyOne.
- We do this to show that the dataset can be explored both globally and by category.



In [15]:
# ------------------------------------------------------------
# STEP 14) Create a filtered view for one class
# ------------------------------------------------------------
#
# This step shows how to work with dataset filters in a demo.

airplane_view = dataset.match(fo.ViewField("ground_truth.label") == "airplane")
print("Airplane samples:", len(airplane_view))



Airplane samples: 42


# STEP 15) Compute class distribution

- Counts how many samples belong to each class.
- Gives a quick summary of the dataset composition.
- We do this to better understand the dataset and add useful context to the demo.



In [21]:
# ------------------------------------------------------------
# STEP 15) Compute class distribution
# ------------------------------------------------------------
#
# This is useful for explaining the dataset composition.

class_counts = dataset.count_values("ground_truth.label")
print("Class distribution:")
for class_name, count in sorted(class_counts.items()):
    print(f"{class_name}: {count}")



Class distribution:
airplane: 42
automobile: 49
bird: 63
cat: 41
deer: 49
dog: 61
frog: 50
horse: 51
ship: 60
truck: 34


# STEP 16) Example: find similar images for a random sample

- Repeats the retrieval process using another sample.
- Shows that similarity search works for more than one fixed example.
- We do this to make the demonstration feel more dynamic and realistic.



In [23]:
# ------------------------------------------------------------
# STEP 16) Example: find similar images for a random sample
# ------------------------------------------------------------
#
# This shows how to repeat the retrieval process with another sample.

random_sample = dataset.take(1).first()
print("Random sample ID:", random_sample.id)
print("Random sample label:", random_sample.ground_truth.label)

random_neighbors = similarity_index.sort_by_similarity(
    random_sample.id,
    k=16,
    dist_field="distance",
)

Random sample ID: 69d849327cb6983a1f2fc4bd
Random sample label: airplane


# STEP 17) Save query metadata into sample fields

- Adds a custom field to each sample and saves it into the dataset.
- Demonstrates how dataset records can be enriched with extra metadata.
- We do this to show that FiftyOne supports both analysis and annotation workflows.



In [24]:
# ------------------------------------------------------------
# STEP 17) Save query metadata into sample fields
# ------------------------------------------------------------
#
# This step demonstrates how to annotate data for later inspection.

for sample in dataset:
    sample["class_name"] = sample.ground_truth.label
    sample.save()

print("Custom field 'class_name' saved into dataset")



Custom field 'class_name' saved into dataset


# STEP 18) Create a mistakes-style demo by comparing top neighbor labels

- Compares each sample with its nearest retrieved neighbor.
- Marks whether the retrieval result matches the sample’s class label.
- We do this to create a simple retrieval quality analysis for demonstration purposes.



In [25]:
# ------------------------------------------------------------
# STEP 18) Create a mistakes-style demo by comparing top neighbor labels
# ------------------------------------------------------------
#
# This is not a true classifier error analysis.
# It is a simple retrieval consistency demo:
# - if the top retrieved neighbor has a different label,
#   we mark it as a retrieval mismatch.

mismatch_ids = []

for sample in dataset.take(50):
    result_view = similarity_index.sort_by_similarity(
        sample.id,
        k=2,
    )

    result_samples = list(result_view)

    if len(result_samples) < 2:
        continue

    # The first result is usually the sample itself.
    # The second one is the nearest neighbor.
    neighbor = result_samples[1]

    sample_label = sample.ground_truth.label
    neighbor_label = neighbor.ground_truth.label

    sample["nearest_neighbor_label"] = neighbor_label
    sample["retrieval_match"] = (sample_label == neighbor_label)
    sample.save()

    if sample_label != neighbor_label:
        mismatch_ids.append(sample.id)

print("Retrieval mismatch analysis completed")
print("Mismatch count:", len(mismatch_ids))



Retrieval mismatch analysis completed
Mismatch count: 7


# STEP 19) Create a mismatch view

- Builds a filtered view containing only retrieval mismatches.
- Helps isolate interesting failure cases for inspection.
- We do this because analyzing mismatches makes the demo more insightful.




In [26]:
# ------------------------------------------------------------
# STEP 19) Create a mismatch view
# ------------------------------------------------------------
#
# This gives you something interesting to demonstrate in FiftyOne.

mismatch_view = dataset.match(fo.ViewField("retrieval_match") == False)

print("Mismatch view size:", len(mismatch_view))



Mismatch view size: 7


# STEP 20) Send mismatch view to the App

- Pushes the mismatch-only subset into the FiftyOne App.
- Makes it easy to visually inspect where retrieval did not behave as expected.
- We do this to focus attention on the most informative examples.


In [27]:
# ------------------------------------------------------------
# STEP 20) Send mismatch view to the App
# ------------------------------------------------------------

session.view = mismatch_view



# STEP 21) Optional: inspect 2D visualization in App

- Encourages exploration of the embedding projection created earlier.
- Lets you visually observe clusters, class groupings, and outliers.
- We do this to complement nearest-neighbor retrieval with a broader view of the feature space.


In [28]:
# ------------------------------------------------------------
# STEP 21) Optional: inspect 2D visualization in App
# ------------------------------------------------------------
#
# In the FiftyOne App:
# - open the "Brain" panel
# - choose "clip_umap"
# - explore clusters visually
#
# This is a nice workshop/demo step.

print("Open the FiftyOne App and inspect the 'clip_umap' visualization")



Open the FiftyOne App and inspect the 'clip_umap' visualization



# STEP 22) Example text prompts with CLIP

- Clarifies that this version keeps the project focused on image-to-image similarity.
- Avoids adding extra complexity that could reduce notebook stability.
- We do this to keep the demo simple, reliable, and easy to explain.


In [29]:
# ------------------------------------------------------------
# STEP 22) Example text prompts with CLIP
# ------------------------------------------------------------
#
# Some FiftyOne CLIP models support prompt-based similarity methods externally,
# but to keep this project stable and simple in Colab,
# we stay with image-to-image similarity only.
#
# That makes the notebook much safer than the earlier large-model setup.

print("Project setup complete")



Project setup complete



# STEP 23) Helpful demo narration points

- Summarizes the main ideas that can be explained during the presentation.
- Connects the technical steps into a clear demonstration story.
- We do this so the project is easier to present confidently.


In [30]:
# ------------------------------------------------------------
# STEP 23) Helpful demo narration points
# ------------------------------------------------------------
#
# You can use these points during your demonstration:
#
# - We loaded a public image dataset
# - We used CLIP to convert each image into an embedding vector
# - Similar images are retrieved by comparing embeddings
# - UMAP gives us a 2D projection for visual exploration
# - Retrieval mismatches help us inspect where semantic confusion happens
#
# This gives a full end-to-end visual AI demo without loading huge models.




# STEP 24) Optional cleanup

- Removes the dataset from FiftyOne storage after the demo if needed.
- Helps keep the environment clean when the project is finished.
- We do this as an optional maintenance step for repeated notebook use.

In [31]:
# ------------------------------------------------------------
# STEP 24) Optional cleanup
# ------------------------------------------------------------
#
# Run this only when you are done and want to remove the dataset.
#
# fo.delete_dataset(dataset_name)

print("Notebook finished successfully")

Notebook finished successfully
